# AI4AM visualizations

Merged MatterGen abstract and poster analyses.


In [ ]:
from __future__ import annotations

import gzip
import json
import sys
from collections.abc import Iterable
from functools import cache
from importlib import resources
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymatgen.analysis.prototypes as prototypes
from IPython.display import Markdown, display
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatviz import structure_2d
from tabulate import tabulate

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    notebooks_dir = path / "notebooks"
    if (notebooks_dir / "notebook_utils.py").exists():
        if str(notebooks_dir) not in sys.path:
            sys.path.insert(0, str(notebooks_dir))
        break
else:
    raise RuntimeError("Could not find notebooks directory")

from notebook_utils import find_repo_root  # noqa: E402

ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
for import_path in (ROOT, NOTEBOOKS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from notebook_constants import (  # noqa: E402
    CRYSTAL_SYSTEM_ORDER,
    SIMPLE_CATEGORY_LABELS,
    SIMPLE_CATEGORY_ORDER,
    WYCKOFF_REPR_FILE,
)
from notebook_utils import (  # noqa: E402
    classify_model,
    crystal_system_from_spg_num,
    load_direct_matches,
    load_pickle_gz,
    missing_required_paths,
    required_paths,
    substituted_relax_info_is_complete,
)
from plot_style import (  # noqa: E402
    CATEGORY_COLORS,
    GRAY,
    WHITE,
    apply_plot_style,
)

from src.config import ANALYSIS_RESULTS_DIR, INPUT_DIR, RAW_RESULTS_DIR  # noqa: E402
from src.sm_anon import AnonMatch  # noqa: E402

MODEL = "mattergen"
AI4AM_DIR = ANALYSIS_RESULTS_DIR / "ai4am"
STRUCTURE_OUT_DIR = AI4AM_DIR / "poster_structures"
SIMPLE_CATEGORY_COLORS = {
    "1": CATEGORY_COLORS["1"],
    "2": CATEGORY_COLORS["2-1"],
    "3": CATEGORY_COLORS["3"],
}

EXAMPLE_ENTRY_IDX = 20297
EXAMPLE_GEN_IDX = 8035
EXAMPLE_TRAIN_IDX = 3438

POSTER_GEN_IDX = 3182

AI4AM_PATH_KEYS = (
    "generated_structures",
    "training_structures",
    "relax_infos",
    "relaxed_ehull",
    "smact_validity",
    "direct_sm",
    "sm_anon_matches",
    "wyckoff_matches",
    "top3_sm_anon",
    "top3_wyckoff",
    "relaxed_sm_anon_entries",
    "relaxed_wyckoff_entries",
    "relaxed_sm_anon_matches",
    "relaxed_wyckoff_matches",
    "wyckoff_repr",
)

paths = required_paths(MODEL, INPUT_DIR, RAW_RESULTS_DIR, AI4AM_PATH_KEYS)
missing_paths = pd.DataFrame(missing_required_paths(paths))
if not missing_paths.empty:
    display(missing_paths)
    raise FileNotFoundError(
        f"Missing required files for MODEL={MODEL!r}. "
        "Check INPUT_DIR and RAW_RESULTS_DIR."
    )

apply_plot_style()
paths

In [ ]:
generated_structures: list[Structure] = load_pickle_gz(paths["generated_structures"])
training_structures: list[Structure] = load_pickle_gz(paths["training_structures"])
wyckoff_repr = load_pickle_gz(paths["wyckoff_repr"])
train_wyckoff_repr = load_pickle_gz(RAW_RESULTS_DIR / "train" / WYCKOFF_REPR_FILE)
if len(train_wyckoff_repr) != len(training_structures):
    raise ValueError("Training Wyckoff data length does not match structures.")

relaxed_sm_anon_records = load_pickle_gz(paths["relaxed_sm_anon_matches"])
relaxed_wyckoff_records = load_pickle_gz(paths["relaxed_wyckoff_matches"])
sm_anon_matches = load_pickle_gz(paths["sm_anon_matches"])
direct_matches = load_direct_matches(paths["direct_sm"])

WYCKOFF_PARAMS_PATH = resources.files(prototypes).joinpath(
    "wyckoff-position-params.json.gz"
)
with gzip.open(WYCKOFF_PARAMS_PATH, "rt") as file:
    WYCKOFF_POSITION_PARAMS = json.load(file)

## Metastable SMACT-valid category ratios

Category ratios by crystal system for MatterGen samples with relaxed hull energy <= 0.1 eV/atom and SMACT-valid composition.


In [ ]:
classifications = classify_model(
    MODEL,
    paths,
    include_crystal_system=True,
    include_simple_category=True,
    simple_category_labels=SIMPLE_CATEGORY_LABELS,
    simple_category_order=SIMPLE_CATEGORY_ORDER,
)
selected_classifications = classifications[
    classifications["is_metastable_smact_valid"]
].copy()
classifications.head()

In [ ]:
def build_crystal_system_category_ratios(
    selected: pd.DataFrame,
) -> pd.DataFrame:
    if selected.empty:
        return pd.DataFrame(
            columns=["crystal_system", "simple_category", "count", "ratio"]
        )

    counts = (
        selected.groupby(["crystal_system", "simple_category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [CRYSTAL_SYSTEM_ORDER, SIMPLE_CATEGORY_ORDER],
        names=["crystal_system", "simple_category"],
    )
    result = counts.reindex(full_index, fill_value=0).reset_index()
    totals = result.groupby("crystal_system", observed=False)["count"].transform("sum")
    result["ratio"] = np.where(totals > 0, result["count"] / totals, 0.0)
    result["simple_category_label"] = result["simple_category"].map(
        SIMPLE_CATEGORY_LABELS
    )
    return result


def build_tabulated_crystal_system_ratios(counts_by_system: pd.DataFrame) -> str:
    if counts_by_system.empty:
        return "No classifications available."

    count_table = (
        counts_by_system.pivot(
            index="crystal_system", columns="simple_category", values="count"
        )
        .reindex(index=CRYSTAL_SYSTEM_ORDER, columns=SIMPLE_CATEGORY_ORDER)
        .fillna(0)
        .astype(int)
    )
    totals = count_table.sum(axis=1)
    count_table = count_table[totals > 0]
    totals = totals[totals > 0]
    ratio_table = count_table.div(totals, axis=0)

    display_table = pd.DataFrame(index=count_table.index)
    for category in SIMPLE_CATEGORY_ORDER:
        display_table[SIMPLE_CATEGORY_LABELS[category]] = [
            f"{count:,} ({ratio:.3f})"
            for count, ratio in zip(
                count_table[category], ratio_table[category], strict=True
            )
        ]
    display_table.insert(0, "crystal_system", display_table.index)
    display_table = display_table.reset_index(drop=True)
    return tabulate(display_table, headers="keys", tablefmt="github", showindex=False)


def plot_category_ratios_by_crystal_system(
    counts_by_system: pd.DataFrame,
    *,
    figsize: tuple[float, float],
    filename: str,
    title: str | None = None,
    rotation: float = 0,
    ha: str = "center",
) -> None:
    if counts_by_system.empty:
        display(Markdown("No classifications available."))
        return

    values = (
        counts_by_system.pivot(
            index="crystal_system", columns="simple_category", values="ratio"
        )
        .reindex(index=CRYSTAL_SYSTEM_ORDER, columns=SIMPLE_CATEGORY_ORDER)
        .fillna(0.0)
    )
    totals = (
        counts_by_system.groupby("crystal_system", observed=False)["count"]
        .sum()
        .reindex(CRYSTAL_SYSTEM_ORDER, fill_value=0)
    )
    systems_with_data = totals[totals > 0].index.tolist()
    if not systems_with_data:
        display(Markdown("No classifications available."))
        return

    values = values.loc[systems_with_data]
    fig, ax = plt.subplots(figsize=figsize)
    bottom = np.zeros(len(values), dtype=float)
    x = np.arange(len(values))
    for category in SIMPLE_CATEGORY_ORDER:
        heights = values[category].to_numpy(dtype=float)
        ax.bar(
            x,
            heights,
            bottom=bottom,
            color=SIMPLE_CATEGORY_COLORS[category],
            edgecolor=WHITE,
            linewidth=0.5,
            label=SIMPLE_CATEGORY_LABELS[category],
        )
        bottom += heights

    if title is not None:
        ax.set_title(title, fontsize=20, pad=40)
    ax.set_ylabel("Ratio", fontsize=20)
    ax.set_ylim(0, 1)
    ax.set_xticks(x, systems_with_data, rotation=rotation, ha=ha)
    ax.tick_params(axis="both", labelsize=16)
    ax.grid(axis="y", color=GRAY, alpha=0.35, linewidth=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 0.98),
        frameon=False,
        fontsize=16,
        ncol=3,
    )
    fig.tight_layout()
    AI4AM_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(AI4AM_DIR / filename, bbox_inches="tight")
    plt.show()


crystal_system_category_ratios = build_crystal_system_category_ratios(
    selected_classifications
)
display(Markdown(build_tabulated_crystal_system_ratios(crystal_system_category_ratios)))
plot_category_ratios_by_crystal_system(
    crystal_system_category_ratios,
    figsize=(8.0, 6.0),
    filename="bar_chart_abstract.pdf",
    title=(r"MatterGen samples ($E_\mathrm{hull} \leq 0.1$ [eV/atom], SMACT-valid)"),
    rotation=30,
    ha="right",
)

In [ ]:
display(Markdown(build_tabulated_crystal_system_ratios(crystal_system_category_ratios)))
plot_category_ratios_by_crystal_system(
    crystal_system_category_ratios,
    figsize=(12.0, 6.0),
    filename="bar_chart_poster.pdf",
)

## MatterGen Wyckoff multiset coverage by space group

Coverage of metastable SMACT-valid MatterGen structures relative to theoretical Wyckoff-letter multisets with at most 20 atoms per primitive unit cell. Bars are indexed by space group number; crystal-system labels mark contiguous space-group ranges for readability.

In [ ]:
from collections import Counter, defaultdict

from notebook_constants import METASTABLE_EHULL_MAX  # noqa: E402
from plot_style import BLACK, PALETTE  # noqa: E402
from pymatgen.analysis.prototypes import (
    WYCKOFF_MULTIPLICITY_DICT,
    WYCKOFF_POSITION_RELAB_DICT,
)
from pymatgen.symmetry.groups import sg_symbol_from_int_number


def add_stacked_bars(
    ax,
    x: np.ndarray,
    values: pd.DataFrame,
    columns: list[str],
    colors: dict[str, str],
    labels: dict[str, str],
    *,
    width: float = 0.8,
    edgecolor: str | None = None,
    linewidth: float = 0.0,
    hatches: dict[str, str] | None = None,
    antialiased: bool | None = None,
    rasterized: bool | None = None,
    snap: bool | None = None,
) -> tuple[dict[str, object], np.ndarray]:
    bottom = np.zeros(len(values), dtype=float)
    legend_handles = {}
    for column in columns:
        heights = values[column].to_numpy(dtype=float)
        bar_edgecolor = colors[column] if edgecolor == "face" else edgecolor
        legend_handles[column] = ax.bar(
            x,
            heights,
            bottom=bottom,
            width=width,
            color=colors[column],
            edgecolor=bar_edgecolor,
            linewidth=linewidth,
            hatch=(hatches or {}).get(column, ""),
            label=labels[column],
            antialiased=antialiased,
            rasterized=rasterized,
            snap=snap,
        )
        bottom += heights
    return legend_handles, bottom


WYCKOFF_MAX_ATOMS = 20
MATTERGEN_WYCKOFF_COVERAGE_MODEL = "mattergen_80000"
WYCKOFF_COVERAGE_COLUMNS = [
    "both",
    "train_only",
    "model_only",
    "icsd",
    "theoretical_only",
]
WYCKOFF_COVERAGE_LABELS = {
    "both": "MP20 & MatterGen",
    "train_only": "MP20 only",
    "model_only": "MatterGen only",
    "icsd": "ICSD",
    "theoretical_only": "Theoretical",
}
WYCKOFF_COVERAGE_COLORS = {
    "both": PALETTE[0],
    "train_only": PALETTE[1],
    "model_only": PALETTE[2],
    "icsd": PALETTE[3],
    "theoretical_only": GRAY,
}


@cache
def wyckoff_relabelings_for_spg(spg_num: int) -> list[dict[int, str]]:
    return WYCKOFF_POSITION_RELAB_DICT.get(str(spg_num), []) or [
        {ord(letter): letter for letter in WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]}
    ]


def wyckoff_relabeling_cycles(
    spg_num: int, trans: dict[int, str]
) -> list[tuple[str, ...]]:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    seen: set[str] = set()
    cycles: list[tuple[str, ...]] = []
    for letter in sorted(multiplicities):
        if letter in seen:
            continue
        cycle = []
        current = letter
        while current not in seen:
            if current not in multiplicities:
                raise ValueError(
                    f"SG {spg_num} relabeling maps to unknown letter {current!r}"
                )
            seen.add(current)
            cycle.append(current)
            current = current.translate(trans)
        cycles.append(tuple(cycle))
    return cycles


WYCKOFF_CENTERING_FACTORS = {
    "P": 1,
    "A": 2,
    "B": 2,
    "C": 2,
    "I": 2,
    "R": 3,
    "F": 4,
}


@cache
def wyckoff_centering_factor_for_spg(spg_num: int) -> int:
    symbol = sg_symbol_from_int_number(spg_num)
    centering = symbol[0]
    if centering not in WYCKOFF_CENTERING_FACTORS:
        raise ValueError(f"SG {spg_num} has unknown centering {centering!r}")
    return WYCKOFF_CENTERING_FACTORS[centering]


@cache
def wyckoff_letter_atom_weight(spg_num: int, letter: str) -> int:
    multiplicity = int(WYCKOFF_MULTIPLICITY_DICT[str(spg_num)][letter])
    factor = wyckoff_centering_factor_for_spg(spg_num)
    if multiplicity % factor != 0:
        raise ValueError(
            f"SG {spg_num} Wyckoff {letter!r} multiplicity {multiplicity} "
            f"is not divisible by centering factor {factor}"
        )
    return multiplicity // factor


@cache
def wyckoff_letter_max_counts(
    spg_num: int, max_atoms: int = WYCKOFF_MAX_ATOMS
) -> dict[str, int]:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    position_params = WYCKOFF_POSITION_PARAMS[str(spg_num)]
    max_counts = {}
    for letter in multiplicities:
        n_free = int(position_params.get(letter, 0))
        weight = wyckoff_letter_atom_weight(spg_num, letter)
        max_counts[letter] = max_atoms // weight if n_free > 0 else 1
    return max_counts


def count_bounded_weighted_solutions_leq(
    weights: list[int], bounds: list[int], max_atoms: int
) -> int:
    dp = [0] * (max_atoms + 1)
    dp[0] = 1
    for weight, bound in zip(weights, bounds, strict=True):
        next_dp = [0] * (max_atoms + 1)
        for total, ways in enumerate(dp):
            if ways == 0:
                continue
            for count in range(bound + 1):
                next_total = total + count * weight
                if next_total > max_atoms:
                    break
                next_dp[next_total] += ways
        dp = next_dp
    return sum(dp[1:])


@cache
def theoretical_wyckoff_count_for_spg(
    spg_num: int, max_atoms: int = WYCKOFF_MAX_ATOMS
) -> int:
    max_counts = wyckoff_letter_max_counts(spg_num, max_atoms)
    relabelings = wyckoff_relabelings_for_spg(spg_num)
    fixed_total = 0
    for trans in relabelings:
        cycles = wyckoff_relabeling_cycles(spg_num, trans)
        cycle_weights = [
            sum(wyckoff_letter_atom_weight(spg_num, letter) for letter in cycle)
            for cycle in cycles
        ]
        cycle_bounds = [min(max_counts[letter] for letter in cycle) for cycle in cycles]
        fixed_total += count_bounded_weighted_solutions_leq(
            cycle_weights, cycle_bounds, max_atoms
        )
    if fixed_total % len(relabelings) != 0:
        raise ValueError(f"SG {spg_num} Burnside count is not integral")
    return fixed_total // len(relabelings)


def build_theoretical_wyckoff_counts() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "spg_num": spg_num,
            "crystal_system": crystal_system_from_spg_num(spg_num),
            "theoretical": theoretical_wyckoff_count_for_spg(spg_num),
        }
        for spg_num in range(1, 231)
    )


def wyckoff_key_atom_count(spg_num: int, key: tuple[str, ...]) -> int:
    return sum(wyckoff_letter_atom_weight(spg_num, letter) for letter in key)


def wyckoff_key_occupancy_violations(
    spg_num: int, key: tuple[str, ...], max_atoms: int = WYCKOFF_MAX_ATOMS
) -> list[str]:
    counts = Counter(key)
    max_counts = wyckoff_letter_max_counts(spg_num, max_atoms)
    return sorted(
        letter for letter, count in counts.items() if count > max_counts[letter]
    )


def canonical_observed_wyckoff_key(data) -> tuple[int, tuple[str, ...]]:
    if not data.letter_key:
        raise ValueError("WyckoffData has no letter_key entries")
    spg_num = int(data.spg_num)
    key = min(tuple(letter_key) for letter_key in data.letter_key)
    return spg_num, key


def observed_wyckoff_sets_by_spg(
    data,
    label: str,
    max_atoms: int = WYCKOFF_MAX_ATOMS,
    *,
    include_indices: set[int] | None = None,
) -> dict[int, set[tuple[str, ...]]]:
    by_spg: dict[int, set[tuple[str, ...]]] = defaultdict(set)
    dropped_atom_count = 0
    dropped_occupancy = 0
    skipped_by_filter = 0
    skipped_none = 0
    for idx, record in enumerate(data):
        if include_indices is not None and idx not in include_indices:
            skipped_by_filter += 1
            continue
        if record is None:
            skipped_none += 1
            continue
        spg_num, key = canonical_observed_wyckoff_key(record)
        multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
        invalid_letters = sorted(set(key) - set(multiplicities))
        if invalid_letters:
            raise ValueError(
                f"{label} has invalid SG {spg_num} letters {invalid_letters}"
            )
        if wyckoff_key_atom_count(spg_num, key) > max_atoms:
            dropped_atom_count += 1
            continue
        if wyckoff_key_occupancy_violations(spg_num, key, max_atoms):
            dropped_occupancy += 1
            continue
        by_spg[spg_num].add(key)
    print(
        f"{label}: {len(data):,} records, "
        f"{sum(len(v) for v in by_spg.values()):,} "
        f"unique keys <= {max_atoms} primitive atoms, "
        f"{skipped_by_filter:,} skipped by subset filter, "
        f"{skipped_none:,} skipped None, "
        f"{dropped_atom_count:,} dropped by atom count, "
        f"{dropped_occupancy:,} dropped by occupancy limits",
        flush=True,
    )
    return dict(by_spg)


def first_metastable_smact_valid_indices_for_wyckoff_coverage(
    model: str, target_count: int, n_wyckoff_records: int
) -> set[int]:
    paths = required_paths(model, INPUT_DIR, RAW_RESULTS_DIR)
    ehull_relaxed = np.asarray(load_pickle_gz(paths["relaxed_ehull"]), dtype=float)
    relax_infos = load_pickle_gz(paths["relax_infos"])
    with np.load(paths["smact_validity"]) as data:
        if "valid" not in data.files:
            raise ValueError(f"{paths['smact_validity']} does not contain 'valid'")
        smact_valid = np.asarray(data["valid"], dtype=bool)

    lengths = {
        "relaxed_ehull": len(ehull_relaxed),
        "relax_infos": len(relax_infos),
        "smact_validity": len(smact_valid),
        "wyckoff_repr": n_wyckoff_records,
    }
    if len(set(lengths.values())) != 1:
        raise ValueError(f"{model} Wyckoff coverage inputs differ in length: {lengths}")

    relax_converged = np.asarray(
        [
            info is not None and bool(info.get("converged", False))
            for info in relax_infos
        ],
        dtype=bool,
    )
    eligible = np.flatnonzero(
        relax_converged
        & np.isfinite(ehull_relaxed)
        & (ehull_relaxed <= METASTABLE_EHULL_MAX)
        & smact_valid
    )
    if len(eligible) < target_count:
        raise ValueError(
            f"{model} has {len(eligible):,} metastable SMACT-valid samples; "
            f"need {target_count:,}."
        )

    selected = eligible[:target_count]
    print(
        f"{model}: selected first {len(selected):,} of {len(eligible):,} "
        "metastable SMACT-valid samples for Wyckoff coverage",
        flush=True,
    )
    return set(int(idx) for idx in selected)


def build_wyckoff_coverage_table(
    train_sets: dict[int, set[tuple[str, ...]]],
    model_sets: dict[int, set[tuple[str, ...]]],
    icsd_sets: dict[int, set[tuple[str, ...]]],
    theory: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    theory_by_spg = theory.set_index("spg_num")["theoretical"].to_dict()
    for spg_num in range(1, 231):
        train = train_sets.get(spg_num, set())
        model = model_sets.get(spg_num, set())
        both = train & model
        train_only = train - model
        model_only = model - train
        observed_union = train | model
        icsd_set = icsd_sets.get(spg_num, set())
        icsd = icsd_set - observed_union
        union_with_icsd = observed_union | icsd_set
        theoretical = int(theory_by_spg[spg_num])
        theoretical_only = theoretical - len(union_with_icsd)
        if theoretical_only < 0:
            raise ValueError(
                f"Observed train/generated/ICSD union for SG {spg_num} exceeds "
                f"theoretical count: {len(union_with_icsd)} > {theoretical}"
            )
        rows.append(
            {
                "model": "mattergen",
                "subset": "metastable_smact_valid",
                "spg_num": spg_num,
                "crystal_system": crystal_system_from_spg_num(spg_num),
                "theoretical": theoretical,
                "both": len(both),
                "train_only": len(train_only),
                "model_only": len(model_only),
                "icsd": len(icsd),
                "theoretical_only": theoretical_only,
                "observed_union": len(observed_union),
            }
        )
    frame = pd.DataFrame(rows)
    assert (frame[WYCKOFF_COVERAGE_COLUMNS].sum(axis=1) == frame["theoretical"]).all()
    assert (
        frame[["both", "train_only", "model_only"]].sum(axis=1)
        == frame["observed_union"]
    ).all()
    return frame


def ratio_frame(
    frame: pd.DataFrame, columns: list[str], denominator: str
) -> pd.DataFrame:
    ratios = frame.copy()
    safe_denominator = ratios[denominator].where(ratios[denominator] != 0, 1)
    for column in columns:
        ratios[column] = ratios[column] / safe_denominator
    return ratios


@cache
def crystal_system_spans() -> list[tuple[str, int, int]]:
    spans = []
    start = 1
    current = crystal_system_from_spg_num(start)
    for spg_num in range(2, 231):
        system = crystal_system_from_spg_num(spg_num)
        if system != current:
            spans.append((current, start, spg_num - 1))
            start = spg_num
            current = system
    spans.append((current, start, 230))
    return spans


def build_wyckoff_coverage_summary(coverage: pd.DataFrame) -> pd.DataFrame:
    if coverage.empty:
        return pd.DataFrame(
            columns=[
                "crystal_system",
                "theoretical",
                "train and MatterGen",
                "train only",
                "MatterGen only",
                "ICSD",
                "Theoretical",
                "observed union",
                "observed/theoretical",
                "MatterGen-only/theoretical",
            ]
        )

    grouped = (
        coverage.groupby("crystal_system", observed=False)[
            ["theoretical", *WYCKOFF_COVERAGE_COLUMNS, "observed_union"]
        ]
        .sum()
        .reindex(CRYSTAL_SYSTEM_ORDER, fill_value=0)
    )
    grouped = grouped[grouped["theoretical"] > 0]
    summary = grouped.rename(
        columns={
            "both": "train and MatterGen",
            "train_only": "train only",
            "model_only": "MatterGen only",
            "icsd": "ICSD",
            "theoretical_only": "Theoretical",
            "observed_union": "observed union",
        }
    )
    summary["observed/theoretical"] = summary["observed union"] / summary["theoretical"]
    summary["MatterGen-only/theoretical"] = (
        summary["MatterGen only"] / summary["theoretical"]
    )
    summary = summary.reset_index()

    count_columns = [
        "theoretical",
        "train and MatterGen",
        "train only",
        "MatterGen only",
        "ICSD",
        "Theoretical",
        "observed union",
    ]
    summary[count_columns] = summary[count_columns].astype(int)
    return summary


def plot_wyckoff_coverage_by_spg(coverage: pd.DataFrame) -> None:
    if coverage.empty:
        display(Markdown("No MatterGen Wyckoff coverage available."))
        return

    ratios = ratio_frame(coverage, WYCKOFF_COVERAGE_COLUMNS, "theoretical").sort_values(
        "spg_num"
    )
    x = ratios["spg_num"].to_numpy(dtype=int)

    fig, ax = plt.subplots(figsize=(12, 4.5))
    legend_handles, bottom = add_stacked_bars(
        ax,
        x,
        ratios,
        WYCKOFF_COVERAGE_COLUMNS,
        WYCKOFF_COVERAGE_COLORS,
        WYCKOFF_COVERAGE_LABELS,
        width=1.0,
        edgecolor="face",
        linewidth=0.1,
        antialiased=False,
        snap=True,
    )

    assert np.allclose(bottom, 1.0)
    spans = crystal_system_spans()
    centers = [(start + end) / 2 for _, start, end in spans]
    for _, _, end in spans[:-1]:
        ax.axvline(
            end + 0.5,
            color=BLACK,
            alpha=0.75,
            linestyle="--",
            linewidth=0.25,
            zorder=3,
        )

    ax.set_xlim(0.5, 230.5)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Ratio", fontsize=20, labelpad=5)
    ax.set_xlabel("Space group", fontsize=20, labelpad=50)
    ax.set_xticks(centers)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0, pad=4)
    ax.tick_params(axis="y", labelsize=14, pad=4)
    label_offsets = {"triclinic": -0.06, "monoclinic": -0.25}
    for system, start, end in spans:
        ax.text(
            (start + end) / 2,
            label_offsets.get(system, -0.06),
            f"{system}\n{start}-{end}",
            ha="center",
            va="top",
            fontsize=14,
            transform=ax.get_xaxis_transform(),
            clip_on=False,
        )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(
        handles=[legend_handles[column] for column in WYCKOFF_COVERAGE_COLUMNS],
        labels=[WYCKOFF_COVERAGE_LABELS[column] for column in WYCKOFF_COVERAGE_COLUMNS],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.98),
        frameon=False,
        fontsize=14,
        ncol=5,
    )
    fig.tight_layout()

    plot_dir = ANALYSIS_RESULTS_DIR / "ai4am"
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        plot_dir / "wyckoff_coverage_by_spg.png",
        bbox_inches="tight",
    )
    plt.show()


mattergen_wyckoff_theory = build_theoretical_wyckoff_counts()
mattergen_train_wyckoff_path = RAW_RESULTS_DIR / "train" / WYCKOFF_REPR_FILE
mattergen_train_wyckoff_data = train_wyckoff_repr
mattergen_train_wyckoff_sets = observed_wyckoff_sets_by_spg(
    mattergen_train_wyckoff_data,
    mattergen_train_wyckoff_path.parent.name,
)
mattergen_icsd_wyckoff_path = RAW_RESULTS_DIR / "icsd" / WYCKOFF_REPR_FILE
mattergen_icsd_wyckoff_data = load_pickle_gz(mattergen_icsd_wyckoff_path)
mattergen_icsd_wyckoff_sets = observed_wyckoff_sets_by_spg(
    mattergen_icsd_wyckoff_data,
    mattergen_icsd_wyckoff_path.parent.name,
)
mattergen_wyckoff_path = (
    RAW_RESULTS_DIR / MATTERGEN_WYCKOFF_COVERAGE_MODEL / WYCKOFF_REPR_FILE
)
mattergen_wyckoff_data = load_pickle_gz(mattergen_wyckoff_path)
mattergen_metastable_smact_valid_indices = (
    first_metastable_smact_valid_indices_for_wyckoff_coverage(
        MATTERGEN_WYCKOFF_COVERAGE_MODEL,
        len(mattergen_train_wyckoff_data),
        len(mattergen_wyckoff_data),
    )
)
mattergen_wyckoff_sets = observed_wyckoff_sets_by_spg(
    mattergen_wyckoff_data,
    mattergen_wyckoff_path.parent.name,
    include_indices=mattergen_metastable_smact_valid_indices,
)
mattergen_wyckoff_coverage = build_wyckoff_coverage_table(
    mattergen_train_wyckoff_sets,
    mattergen_wyckoff_sets,
    mattergen_icsd_wyckoff_sets,
    mattergen_wyckoff_theory,
)
mattergen_wyckoff_coverage_summary = build_wyckoff_coverage_summary(
    mattergen_wyckoff_coverage
)
display(mattergen_wyckoff_coverage_summary.round(4))
plot_wyckoff_coverage_by_spg(mattergen_wyckoff_coverage)

## Abstract substituted-match example

A space-group 225 or 229 pair from the relaxed Wyckoff substituted-match records with volume ratio >= 1.5, `cost_uniform=1`, exactly two distinct elements in each structure, `cost_mod_petti <= 5`, and no exact StructureMatcher match in `sm_fit.npz`.


In [ ]:
example_records = [
    record
    for record in relaxed_wyckoff_records
    if int(record["entry_idx"]) == EXAMPLE_ENTRY_IDX
]
if len(example_records) != 1:
    raise ValueError(
        f"Expected one record for entry_idx={EXAMPLE_ENTRY_IDX}, "
        f"found {len(example_records)}"
    )
example_record = example_records[0]

assert bool(example_record["match"])
assert int(example_record["gen_idx"]) == EXAMPLE_GEN_IDX
assert int(example_record["train_idx"]) == EXAMPLE_TRAIN_IDX
assert np.isclose(float(example_record["cost_uniform"]), 1.0)
assert float(example_record["cost_mod_petti"]) <= 5.0

example_classification = classifications[
    classifications["gen_idx"] == EXAMPLE_GEN_IDX
].iloc[0]
assert bool(example_classification["is_smact_valid"])
assert bool(example_classification["is_metastable"])
assert not bool(example_classification["is_direct_sm_match"])
assert EXAMPLE_GEN_IDX not in set(direct_matches["gen_idx"])

generated_example = generated_structures[EXAMPLE_GEN_IDX]
training_example = training_structures[EXAMPLE_TRAIN_IDX]
generated_analyzer = SpacegroupAnalyzer(generated_example)
training_analyzer = SpacegroupAnalyzer(training_example)
generated_spg_num = generated_analyzer.get_space_group_number()
training_spg_num = training_analyzer.get_space_group_number()
assert generated_spg_num in (225, 229)
assert training_spg_num in (225, 229)
assert len(generated_example.composition.elements) == 2
assert len(training_example.composition.elements) == 2

volume_ratio = max(generated_example.volume, training_example.volume) / min(
    generated_example.volume, training_example.volume
)
volume_difference = abs(generated_example.volume - training_example.volume)
assert volume_ratio >= 1.5

example_summary = pd.DataFrame(
    [
        {
            "role": "generated",
            "index": EXAMPLE_GEN_IDX,
            "formula": generated_example.composition.reduced_formula,
            "space_group": (
                f"{generated_analyzer.get_space_group_symbol()} ({generated_spg_num})"
            ),
            "atoms": len(generated_example),
            "distinct_elements": len(generated_example.composition.elements),
            "volume": generated_example.volume,
        },
        {
            "role": "train",
            "index": EXAMPLE_TRAIN_IDX,
            "formula": training_example.composition.reduced_formula,
            "space_group": (
                f"{training_analyzer.get_space_group_symbol()} ({training_spg_num})"
            ),
            "atoms": len(training_example),
            "distinct_elements": len(training_example.composition.elements),
            "volume": training_example.volume,
        },
    ]
)

display(
    Markdown(
        f"**entry_idx:** `{EXAMPLE_ENTRY_IDX}`  \n"
        f"**cost_uniform:** `{float(example_record['cost_uniform']):.6g}`  \n"
        f"**cost_mod_petti:** `{float(example_record['cost_mod_petti']):.6g}`  \n"
        f"**volume ratio:** `{volume_ratio:.6g}`  \n"
        f"**volume difference:** `{volume_difference:.6g}`"
    )
)
display(example_summary)

In [ ]:
def structure_title(role: str, index_label: str, structure: Structure) -> str:
    analyzer = SpacegroupAnalyzer(structure)
    spg = f"{analyzer.get_space_group_symbol()} ({analyzer.get_space_group_number()})"
    return (
        f"{role}<br>{index_label}<br>{structure.composition.reduced_formula}<br>{spg}"
    )


example_structures = {
    "Generated": generated_example,
    "Train": training_example,
}
example_titles = {
    "Generated": structure_title(
        "Generated sample", f"gen_idx={EXAMPLE_GEN_IDX}", generated_example
    ),
    "Train": structure_title(
        "Matched train sample", f"train_idx={EXAMPLE_TRAIN_IDX}", training_example
    ),
}

abstract_example_fig = structure_2d(
    example_structures,
    n_cols=2,
    show_cell=True,
    site_labels="legend",
    standardize_struct=False,
    subplot_title=lambda _struct, key: example_titles[key],
)
abstract_example_fig.update_layout(
    height=360, margin={"l": 10, "r": 10, "t": 80, "b": 10}
)
display(abstract_example_fig)

In [ ]:
def to_conventional_cubic_cell(structure: Structure) -> Structure:
    analyzer = SpacegroupAnalyzer(structure)
    conventional = analyzer.get_conventional_standard_structure()
    if conventional.get_space_group_info()[1] < 195:
        raise ValueError("Expected a cubic conventional cell.")
    return conventional


abstract_generated_conventional = to_conventional_cubic_cell(generated_example)
abstract_training_conventional = to_conventional_cubic_cell(training_example)

abstract_generated_cif_path = AI4AM_DIR / "mattergen_gen_8035_CoO.cif"
abstract_training_cif_path = AI4AM_DIR / "train_3438_MnSe.cif"

AI4AM_DIR.mkdir(parents=True, exist_ok=True)
abstract_generated_conventional.to(filename=abstract_generated_cif_path)
abstract_training_conventional.to(filename=abstract_training_cif_path)

display(
    Markdown(
        f"Wrote CIF files:  \n- `{abstract_generated_cif_path}`  \n"
        f"- `{abstract_training_cif_path}`"
    )
)

## Poster structure example

MatterGen example with a free Wyckoff positional parameter and mixed substituted-relaxed match outcomes among the top 3 `sm_anon` and top 3 Wyckoff candidates.


In [ ]:
wyckoff_matches = load_pickle_gz(paths["wyckoff_matches"])
highest_cost_matches_by_source = {
    "sm_anon": sm_anon_matches,
    "wyckoff": wyckoff_matches,
}

raw_entries_by_source = {
    "sm_anon": load_pickle_gz(paths["top3_sm_anon"]),
    "wyckoff": load_pickle_gz(paths["top3_wyckoff"]),
}
entries_by_source = {
    "sm_anon": load_pickle_gz(paths["relaxed_sm_anon_entries"]),
    "wyckoff": load_pickle_gz(paths["relaxed_wyckoff_entries"]),
}
records_by_source = {
    "sm_anon": relaxed_sm_anon_records,
    "wyckoff": relaxed_wyckoff_records,
}
infos_by_source = {
    "sm_anon": load_pickle_gz(
        paths["relaxed_sm_anon_entries"].with_name(
            paths["relaxed_sm_anon_entries"].name.removesuffix(".pkl.gz")
            + "_infos.pkl.gz"
        )
    ),
    "wyckoff": load_pickle_gz(
        paths["relaxed_wyckoff_entries"].with_name(
            paths["relaxed_wyckoff_entries"].name.removesuffix(".pkl.gz")
            + "_infos.pkl.gz"
        )
    ),
}


def sm_anon_match_lookup(
    gen_idx: int,
    train_indices: Iterable[int],
) -> dict[tuple[int, int], AnonMatch]:
    wanted = {(gen_idx, int(train_idx)) for train_idx in train_indices}
    found: dict[tuple[int, int], AnonMatch] = {}
    for match in sm_anon_matches:
        key = (int(match.idx1), int(match.idx2))
        if key in wanted:
            found[key] = match
            if len(found) == len(wanted):
                break
    missing = wanted - found.keys()
    if missing:
        raise ValueError(f"Missing sm_anon matches for {sorted(missing)}")
    return found


def generated_wyckoff_rows(structure: Structure) -> pd.DataFrame:
    analyzer = SpacegroupAnalyzer(structure, symprec=0.01)
    spg_number = analyzer.get_space_group_number()
    symmetrized = analyzer.get_symmetrized_structure()
    rows = []
    for wyckoff_symbol, sites_group in zip(
        symmetrized.wyckoff_symbols,
        symmetrized.equivalent_sites,
        strict=True,
    ):
        letter = wyckoff_symbol.lstrip("0123456789")
        n_free = int(WYCKOFF_POSITION_PARAMS[str(spg_number)].get(letter, 0))
        rows.append(
            {
                "wyckoff_symbol": wyckoff_symbol,
                "letter": letter,
                "multiplicity_in_cell": len(sites_group),
                "element": sites_group[0].species_string,
                "n_free_parameters": n_free,
                "representative_frac_coords": tuple(
                    float(value) for value in sites_group[0].frac_coords
                ),
            }
        )
    return pd.DataFrame(rows)


def conventional_structure(structure: Structure) -> Structure:
    return SpacegroupAnalyzer(
        structure,
        symprec=0.01,
    ).get_conventional_standard_structure()


def safe_label(value: str) -> str:
    return "".join(char if char.isalnum() else "_" for char in value)


def save_conventional_cif(
    structure: Structure,
    filename: str,
) -> tuple[Structure, Path]:
    conventional = conventional_structure(structure)
    STRUCTURE_OUT_DIR.mkdir(parents=True, exist_ok=True)
    path = STRUCTURE_OUT_DIR / filename
    conventional.to(filename=path)
    return conventional, path


def training_metadata(train_idx: int) -> dict[str, Any]:
    structure = training_structures[train_idx]
    lattice = structure.lattice
    return {
        "train_idx": train_idx,
        "train_formula": structure.composition.reduced_formula,
        "train_spg_num": int(train_wyckoff_repr[train_idx].spg_num),
        "train_a": float(lattice.a),
        "train_b": float(lattice.b),
        "train_c": float(lattice.c),
        "train_alpha": float(lattice.alpha),
        "train_beta": float(lattice.beta),
        "train_gamma": float(lattice.gamma),
        "training_structure": structure,
    }


def highest_cost_rows(source: str, gen_idx: int, k: int = 3) -> pd.DataFrame:
    matches = [
        match
        for match in highest_cost_matches_by_source[source]
        if int(match.idx1) == gen_idx and not np.isnan(float(match.cost_mod_petti))
    ]
    ranked = sorted(
        matches,
        key=lambda match: (-float(match.cost_mod_petti), int(match.idx2)),
    )[:k]
    if len(ranked) != k:
        raise ValueError(
            f"Expected {k} {source} matches for gen_idx={gen_idx}; found {len(ranked)}."
        )

    rows: list[dict[str, Any]] = []
    for rank, match in enumerate(ranked, start=1):
        train_idx = int(match.idx2)
        rows.append(
            {
                "source": source,
                "rank": rank,
                "gen_idx": int(match.idx1),
                **training_metadata(train_idx),
                "cost_uniform": float(match.cost_uniform),
                "cost_mod_petti": float(match.cost_mod_petti),
            }
        )
    return pd.DataFrame(rows)


def highest_cost_training_rows(gen_idx: int, k: int = 3) -> pd.DataFrame:
    return pd.concat(
        [highest_cost_rows(source, gen_idx, k) for source in ("sm_anon", "wyckoff")],
        ignore_index=True,
    )


def record_rows(gen_idx: int) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    sm_anon_train_indices = [
        int(entry.train_idx)
        for entry in entries_by_source["sm_anon"]
        if int(entry.gen_idx) == gen_idx and int(entry.rank) <= 3
    ]
    sm_anon_matches = sm_anon_match_lookup(gen_idx, sm_anon_train_indices)
    for source in ("sm_anon", "wyckoff"):
        for entry_idx, (raw_entry, entry, record, info) in enumerate(
            zip(
                raw_entries_by_source[source],
                entries_by_source[source],
                records_by_source[source],
                infos_by_source[source],
                strict=True,
            )
        ):
            if int(entry.gen_idx) != gen_idx or int(entry.rank) > 3:
                continue
            if int(record["entry_idx"]) != entry_idx:
                raise ValueError(f"{source}: entry_idx mismatch at {entry_idx}")
            for attr in ("gen_idx", "train_idx", "rank"):
                if int(getattr(raw_entry, attr)) != int(getattr(entry, attr)):
                    raise ValueError(
                        f"{source}: raw/relaxed {attr} mismatch at {entry_idx}"
                    )
                if int(getattr(entry, attr)) != int(record[attr]):
                    raise ValueError(f"{source}: {attr} mismatch at {entry_idx}")
            for attr in ("cost_uniform", "cost_mod_petti"):
                if not np.isclose(
                    float(getattr(raw_entry, attr)), float(getattr(entry, attr))
                ):
                    raise ValueError(
                        f"{source}: raw/relaxed {attr} mismatch at {entry_idx}"
                    )
                if not np.isclose(float(getattr(entry, attr)), float(record[attr])):
                    raise ValueError(f"{source}: {attr} mismatch at {entry_idx}")
            train_idx = int(entry.train_idx)
            rows.append(
                {
                    "source": source,
                    "rank": int(entry.rank),
                    "entry_idx": entry_idx,
                    "gen_idx": int(entry.gen_idx),
                    **training_metadata(train_idx),
                    "cost_uniform": float(entry.cost_uniform),
                    "cost_mod_petti": float(entry.cost_mod_petti),
                    "relaxed_match": bool(record["match"])
                    and substituted_relax_info_is_complete(info),
                    "relax_failed": not substituted_relax_info_is_complete(info),
                    "relax_converged": None
                    if info is None
                    else bool(info.get("converged", False)),
                    "sm_anon_match": sm_anon_matches[(int(entry.gen_idx), train_idx)]
                    if source == "sm_anon"
                    else None,
                    "substituted_structure": raw_entry.structure,
                    "relaxed_substituted_structure": entry.structure,
                }
            )
    frame = pd.DataFrame(rows)
    return frame.sort_values(["source", "rank"]).reset_index(drop=True)


example = record_rows(POSTER_GEN_IDX)
highest_cost_training = highest_cost_training_rows(POSTER_GEN_IDX)
generated_structure = generated_structures[POSTER_GEN_IDX]
spg_num = int(wyckoff_repr[POSTER_GEN_IDX].spg_num)
crystal_system = crystal_system_from_spg_num(spg_num)

generated_wyckoff = generated_wyckoff_rows(generated_structure)
free_wyckoff_letters = set(
    generated_wyckoff.loc[
        generated_wyckoff["n_free_parameters"] > 0,
        "letter",
    ]
)

generated_conventional_structure, _ = save_conventional_cif(
    generated_structure,
    f"mattergen_gen_{POSTER_GEN_IDX}_{safe_label(generated_structure.composition.reduced_formula)}"
    "_conventional.cif",
)


def save_example_structures(row) -> dict[str, Structure | Path]:
    key = f"{row.source}_rank{int(row.rank)}_train_{int(row.train_idx)}"
    formula = safe_label(row.train_formula)
    saved = {}
    for role, structure in (
        ("training", row.training_structure),
        ("substituted", row.substituted_structure),
        ("relaxed_substituted", row.relaxed_substituted_structure),
    ):
        conventional, path = save_conventional_cif(
            structure,
            f"{key}_{formula}_{role}_conventional.cif",
        )
        saved[f"{role}_conventional_structure"] = conventional
        saved[f"{role}_conventional_cif"] = path
    return saved


example_cifs = [save_example_structures(row) for row in example.itertuples(index=False)]
example = example.assign(
    training_conventional_structure=[
        saved["training_conventional_structure"] for saved in example_cifs
    ],
    substituted_conventional_structure=[
        saved["substituted_conventional_structure"] for saved in example_cifs
    ],
    relaxed_substituted_conventional_structure=[
        saved["relaxed_substituted_conventional_structure"] for saved in example_cifs
    ],
    training_conventional_cif=[
        saved["training_conventional_cif"] for saved in example_cifs
    ],
    substituted_conventional_cif=[
        saved["substituted_conventional_cif"] for saved in example_cifs
    ],
    relaxed_substituted_conventional_cif=[
        saved["relaxed_substituted_conventional_cif"] for saved in example_cifs
    ],
)


def save_highest_cost_training_structure(row) -> tuple[Structure, Path]:
    formula = safe_label(row.train_formula)
    return save_conventional_cif(
        row.training_structure,
        f"{row.source}_highest_cost_rank{int(row.rank)}_train_{int(row.train_idx)}"
        f"_{formula}_training_conventional.cif",
    )


highest_cost_cifs = [
    save_highest_cost_training_structure(row)
    for row in highest_cost_training.itertuples(index=False)
]
highest_cost_training = highest_cost_training.assign(
    training_conventional_structure=[saved[0] for saved in highest_cost_cifs],
    training_conventional_cif=[saved[1] for saved in highest_cost_cifs],
)

display(
    Markdown(
        f"## Selected MatterGen example  \\n"
        f"gen_idx=`{POSTER_GEN_IDX}`; "
        f"formula=`{generated_structure.composition.reduced_formula}`; "
        f"conventional atoms=`{len(generated_conventional_structure)}`; "
        f"space group=`{spg_num}`; crystal system=`{crystal_system}`; "
        f"free Wyckoff letters=`{sorted(free_wyckoff_letters)}`"
    )
)
display(generated_wyckoff)
display(Markdown("## Highest-cost training matches"))
display(
    highest_cost_training.drop(
        columns=["training_structure", "training_conventional_structure"]
    )
)
display(
    example.drop(
        columns=[
            "sm_anon_match",
            "training_structure",
            "substituted_structure",
            "relaxed_substituted_structure",
            "training_conventional_structure",
            "substituted_conventional_structure",
            "relaxed_substituted_conventional_structure",
        ]
    )
)
display(Markdown(f"## Conventional CIFs saved to `{STRUCTURE_OUT_DIR}`"))

In [ ]:
def status_label(row) -> str:
    if row.relax_failed:
        return "relax failed"
    if row.relaxed_match:
        return "relaxed match"
    return "no relaxed match"


def format_cost(value: Any) -> str:
    if pd.isna(value):
        return "N/A"
    return f"{float(value):.6g}"


def panel_title(row, role: str) -> str:
    return (
        f"{role}<br>{row.source} rank {int(row.rank)}; "
        f"train_idx={int(row.train_idx)}<br>"
        f"{row.train_formula}; {status_label(row)}<br>"
        f"uniform={format_cost(row.cost_uniform)}; "
        f"mod-Petti={format_cost(row.cost_mod_petti)}"
    )


def structure_grid(
    frame: pd.DataFrame,
    structure_column: str,
    key_builder,
    title_builder,
    *,
    height: int = 720,
):
    rows = list(frame.itertuples(index=False))
    structures = {key_builder(row): getattr(row, structure_column) for row in rows}
    titles = {key_builder(row): title_builder(row) for row in rows}
    fig = structure_2d(
        structures,
        n_cols=3,
        show_cell=True,
        standardize_struct=False,
        subplot_title=lambda _struct, key: titles[key],
    )
    fig.update_layout(
        height=height,
        margin={"l": 10, "r": 10, "t": 90, "b": 10},
    )
    return fig


generated_fig = structure_2d(
    {"generated": generated_structure},
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _struct, _key: (
        f"MatterGen generated<br>gen_idx={POSTER_GEN_IDX}; "
        f"{generated_structure.composition.reduced_formula}<br>"
        f"spg={spg_num}; {crystal_system}"
    ),
)
generated_fig.update_layout(height=320, margin={"l": 10, "r": 10, "t": 80, "b": 10})
display(generated_fig)

train_fig = structure_grid(
    example,
    "training_structure",
    lambda row: f"{row.source}_{int(row.rank)}",
    lambda row: panel_title(row, "Training"),
)
display(Markdown("## Matched training structures"))
display(train_fig)


def highest_cost_title(row) -> str:
    return (
        f"{row.source} highest-cost rank {int(row.rank)}<br>"
        f"train_idx={int(row.train_idx)}<br>"
        f"{row.train_formula}<br>"
        f"uniform={format_cost(row.cost_uniform)}; "
        f"mod-Petti={format_cost(row.cost_mod_petti)}"
    )


highest_cost_fig = structure_grid(
    highest_cost_training,
    "training_structure",
    lambda row: f"{row.source}_high_cost_{int(row.rank)}",
    highest_cost_title,
)
display(Markdown("## Highest-cost training structures"))
display(highest_cost_fig)

relaxed_fig = structure_grid(
    example,
    "relaxed_substituted_conventional_structure",
    lambda row: f"{row.source}_{int(row.rank)}",
    lambda row: panel_title(row, "Relaxed substituted"),
)
display(Markdown("## Relaxed substituted training structures"))
display(relaxed_fig)


def sm_anon_commensurate_structure(row) -> Structure:
    match = row.sm_anon_match
    if match is None:
        raise ValueError("sm_anon_match is required")

    generated = generated_structures[int(row.gen_idx)].copy()
    training = row.training_structure.copy()
    if bool(match.s1_supercell):
        generated.make_supercell(match.supercell_matrix)
    else:
        training.make_supercell(match.supercell_matrix)

    if len(generated) != len(training):
        raise ValueError(
            "sm_anon commensurate generated/training structures have different sizes."
        )
    if len(match.mapping) != len(generated):
        raise ValueError(
            "sm_anon mapping length does not match commensurate structure size."
        )
    if min(match.mapping) < 0 or max(match.mapping) >= len(generated):
        raise ValueError("sm_anon mapping contains an out-of-bounds atom index.")
    return generated


sm_anon_rows = example[example["source"] == "sm_anon"].sort_values("rank")
generated_commensurate_structures = {}
generated_commensurate_titles = {}
for row in sm_anon_rows.itertuples(index=False):
    key = f"sm_anon_{int(row.rank)}"
    generated_commensurate = sm_anon_commensurate_structure(row)
    generated_commensurate_structures[key] = generated_commensurate
    generated_commensurate_titles[key] = (
        f"Generated commensurate cell<br>sm_anon rank {int(row.rank)}; "
        f"train_idx={int(row.train_idx)}"
    )

generated_supercell_fig = structure_2d(
    generated_commensurate_structures,
    n_cols=3,
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _struct, key: generated_commensurate_titles[key],
)
generated_supercell_fig.update_layout(
    height=420,
    margin={"l": 10, "r": 10, "t": 90, "b": 10},
)
display(Markdown("## Generated commensurate cells used by sm_anon"))
display(generated_supercell_fig)